# LeetCode 433 — Minimum Genetic Mutation bằng Python

Link đề: https://leetcode.com/problems/minimum-genetic-mutation/ · Độ khó: **Medium** · Chủ đề: Breadth-First Search (BFS), Hash Table, String, Bidirectional Search (tìm kiếm hai chiều), Graph (đồ thị ẩn)

Skeleton LeetCode:

```python
class Solution:
    def minMutation(self, startGene: str, endGene: str, bank: List[str]) -> int:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án đã được kiểm chứng bằng code (đối chiếu năm cách làm với một oracle Floyd–Warshall độc lập trên hàng trăm bộ dữ liệu ngẫu nhiên). Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi gene dài 8 ký tự từ `A, C, G, T`; **một lần đột biến** = đổi **đúng một** ký tự; gene sau đột biến **phải nằm trong `bank`**. Tìm **số lần đột biến ít nhất** từ `startGene` đến `endGene`, hoặc `-1`.
2. Nắm các công cụ: so sánh hai chuỗi từng ký tự bằng `zip`, **đếm số vị trí khác nhau** (khoảng cách Hamming), `set` để tra cứu/đánh dấu **đã thăm** trong `O(1)`, **cắt ghép chuỗi** `s[:i] + c + s[i+1:]`, `deque` làm hàng đợi, và phép giao tập hợp.
3. Đi từ **vét cạn bằng quay lui (DFS)** → **BFS quét ngân hàng** → **BFS sinh láng giềng bằng cách thay từng ký tự** → **BFS hai chiều** → **nới lỏng khoảng cách kiểu Bellman–Ford**.
4. Hiểu mấu chốt: coi mỗi gene là một **đỉnh**, hai gene khác nhau đúng một ký tự là **cạnh**; "số lần đột biến ít nhất" = **đường đi ngắn nhất trên đồ thị không trọng số** ⇒ dùng **BFS**.
5. Tránh các lỗi kinh điển: quên `visited` (lặp vô hạn), nhầm điều kiện "khác đúng một ký tự", quên kiểm tra `endGene` có trong `bank`, trả về sai khi `start == end`, và dùng DFS rồi tưởng đường đầu tiên là ngắn nhất.

## 2. Đề bài & ví dụ

Chuỗi gene là chuỗi dài **8** ký tự gồm `'A'`, `'C'`, `'G'`, `'T'`. Một **đột biến** là đổi **đúng một** ký tự của chuỗi, ví dụ `"AACCGGTT" → "AACCGGTA"`.

Có một **ngân hàng gene** `bank` ghi các gene hợp lệ: **một gene chỉ hợp lệ nếu nằm trong `bank`**. Điểm xuất phát `startGene` được coi là hợp lệ dù **có thể không** nằm trong `bank`.

Cho `startGene`, `endGene`, `bank`, trả về **số đột biến tối thiểu** để biến `startGene` thành `endGene` (mọi gene trung gian **và** `endGene` đều phải nằm trong `bank`); nếu không thể thì trả về `-1`.

**Ví dụ 1:** `startGene = "AACCGGTT"`, `endGene = "AACCGGTA"`, `bank = ["AACCGGTA"]` → `1`.

**Ví dụ 2:** `startGene = "AACCGGTT"`, `endGene = "AAACGGTA"`, `bank = ["AACCGGTA","AACCGCTA","AAACGGTA"]` → `2` (`AACCGGTT → AACCGGTA → AAACGGTA`).

**Ràng buộc:** `0 <= bank.length <= 10`; `startGene.length == endGene.length == bank[i].length == 8`; các chuỗi chỉ gồm `A, C, G, T`.

In [1]:
examples = [
    ("AACCGGTT", "AACCGGTA", ["AACCGGTA"], 1),
    ("AACCGGTT", "AAACGGTA", ["AACCGGTA", "AACCGCTA", "AAACGGTA"], 2),
    ("AAAAACCC", "AACCCCCC", ["AAAACCCC", "AAACCCCC", "AACCCCCC"], 3),    # tự dựng: chuỗi 3 bước
]
for s, e, bank, ans in examples:
    print(s, "->", e, "| bank =", bank, "| kỳ vọng", ans)

AACCGGTT -> AACCGGTA | bank = ['AACCGGTA'] | kỳ vọng 1
AACCGGTT -> AAACGGTA | bank = ['AACCGGTA', 'AACCGCTA', 'AAACGGTA'] | kỳ vọng 2
AAAAACCC -> AACCCCCC | bank = ['AAAACCCC', 'AAACCCCC', 'AACCCCCC'] | kỳ vọng 3


## 3. Kiến thức nền

### 3.1. So sánh hai chuỗi từng ký tự: `zip` và khoảng cách Hamming

`zip(a, b)` ghép từng cặp ký tự **cùng vị trí** của hai chuỗi. Đếm số cặp **khác nhau** cho ta **khoảng cách Hamming** — số vị trí hai chuỗi (cùng độ dài) khác nhau. Một đột biến hợp lệ ⇔ khoảng cách Hamming **đúng bằng 1**. (Bằng `0` là cùng một gene, bằng `2` trở lên là cần ≥ 2 đột biến.)

In [2]:
def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))

print(list(zip("ACGT", "AGGA")))                 # [('A','A'), ('C','G'), ('G','G'), ('T','A')]
print(hamming("AACCGGTT", "AACCGGTA"))           # 1  -> một đột biến
print(hamming("AACCGGTT", "AAACGGTA"))           # 2  -> cần ít nhất hai bước
print(hamming("AACCGGTT", "AACCGGTT"))           # 0  -> cùng gene

[('A', 'A'), ('C', 'G'), ('G', 'G'), ('T', 'A')]
1
2
0


### 3.2. Đồ thị ẩn: gene = đỉnh, đột biến = cạnh

Ta không được cho sẵn đồ thị, nhưng có thể **nhìn ra** nó: mỗi gene (trong `bank` hoặc `startGene`) là một **đỉnh**; có **cạnh** giữa hai gene nếu khác nhau đúng một ký tự. Một chuỗi đột biến chính là một **đường đi** trên đồ thị này; "ít đột biến nhất" = **đường đi có ít cạnh nhất**.

Ta dựng thử đồ thị cho ví dụ 2 để *nhìn thấy* nó:

In [3]:
start, end, bank, _ = examples[1]
nodes = [start] + bank
adj = {g: [h for h in nodes if hamming(g, h) == 1] for g in nodes}
for g, nb in adj.items():
    print(g, "->", nb)

AACCGGTT -> ['AACCGGTA']
AACCGGTA -> ['AACCGGTT', 'AACCGCTA', 'AAACGGTA']
AACCGCTA -> ['AACCGGTA']
AAACGGTA -> ['AACCGGTA']


### 3.3. BFS tìm đường ngắn nhất trên đồ thị **không trọng số**

**Hàng đợi** (queue) là cấu trúc vào-trước-ra-trước (FIFO). **BFS**: xếp đỉnh xuất phát vào hàng đợi; lặp: lấy đỉnh ở đầu hàng ra, đưa các láng giềng **chưa thăm** vào cuối hàng. Vì ta mở rộng **theo từng "vòng"** (tất cả đỉnh cách `k` bước trước rồi mới tới `k+1` bước), **lần đầu** gặp một đỉnh chính là lúc ta đến nó bằng đường **ngắn nhất**. Trong Python dùng `collections.deque` (`append`, `popleft` đều `O(1)`).

In [4]:
from collections import deque

q = deque([("S", 0)])
q.append(("A", 1)); q.append(("B", 1))
print(q.popleft(), q.popleft(), len(q))      # ('S', 0) ('A', 1) 1

('S', 0) ('A', 1) 1


### 3.4. `set` để đánh dấu "đã thăm" và tra cứu nhanh

`x in s` với `set` là `O(1)` trung bình, còn với `list` là `O(độ dài)`. Ta dùng `set` cho hai việc: `bank_set` (gene có hợp lệ không) và `visited` (đã đưa vào hàng đợi chưa — để **không bao giờ** xử lý lại một gene, tránh lặp vô hạn trên đồ thị có chu trình).

In [5]:
bank_set = set(["AACCGGTA", "AAACGGTA"])
print("AACCGGTA" in bank_set, "AACCGGTC" in bank_set)     # True False
visited = {"AACCGGTT"}
visited.add("AACCGGTA")
print(len(visited), "AACCGGTA" in visited)                # 2 True

True False
2 True


### 3.5. Sinh láng giềng bằng cách **thay một ký tự**

Thay vì so từng gene của `bank` với gene hiện tại, ta có thể **tự tạo** mọi đột biến một ký tự: với mỗi vị trí `i` (`0..7`) và mỗi chữ cái `c` trong `"ACGT"` khác `gene[i]`, tạo `gene[:i] + c + gene[i+1:]` (cắt chuỗi: phần trước `i`, chữ cái mới, phần sau `i`). Mỗi gene có đúng `8 × 3 = 24` đột biến có thể; giữ lại cái nào **nằm trong `bank_set`**.

In [6]:
def neighbors(gene, bank_set):
    res = []
    for i in range(len(gene)):
        for c in "ACGT":
            if c != gene[i]:
                cand = gene[:i] + c + gene[i + 1:]       # đổi đúng ký tự thứ i
                if cand in bank_set:
                    res.append(cand)
    return res

print(len("AACCGGTT") * 3)                                      # 24 ứng viên
print(neighbors("AACCGGTT", set(examples[1][2])))               # ['AACCGGTA']
print(neighbors("AACCGGTA", set(examples[1][2])))               # các gene trong bank cách 1 ký tự

24
['AACCGGTA']
['AAACGGTA', 'AACCGCTA']


### 3.6. Phép giao tập hợp `&` và tìm kiếm hai chiều

Với `set` ta có `a & b` (phần giao) và `a | b`. **BFS hai chiều** chạy **đồng thời từ hai đầu** (từ `start` và từ `end`) và dừng khi hai "mặt trận" gặp nhau. Hai mặt trận mỗi bên đi một nửa quãng đường nên số đỉnh phải duyệt thường nhỏ hơn nhiều.

In [7]:
front, back = {"x", "y"}, {"y", "z"}
print(front & back)           # {'y'}  -> hai mặt trận gặp nhau
print(bool(front & back))     # True

{'y'}
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 3 (tự dựng): `start = AAAAACCC`, `end = AACCCCCC`, `bank = [AAAACCCC, AAACCCCC, AACCCCCC]`.

- `hamming(start, AAAACCCC) = 1` (vị trí 4: `A→C`) ⇒ một đột biến tới `AAAACCCC` (có trong bank).
- `AAAACCCC → AAACCCCC` (vị trí 3: `A→C`): một đột biến, có trong bank.
- `AAACCCCC → AACCCCCC` (vị trí 2: `A→C`): một đột biến tới `end`.

Còn `start → AAACCCCC` khác 2 ký tự ⇒ **không** đi một bước được. Vậy đường duy nhất: `start → AAAACCCC → AAACCCCC → end`: **3** đột biến.

Tính tay cho ví dụ 2: từ `AACCGGTT`, chỉ có `AACCGGTA` cách đúng 1 ký tự (khác ở vị trí cuối) ⇒ **vòng 1**: `{AACCGGTA}`. Từ `AACCGGTA`: `AACCGCTA` (khác vị trí 5) và `AAACGGTA` (khác vị trí 2) đều cách 1 ký tự ⇒ **vòng 2**: `{AACCGCTA, AAACGGTA}` và ta thấy `end = AAACGGTA` ⇒ đáp án **2**.

Ý chính: **đi theo vòng**; vòng nào gặp `end` đầu tiên thì số vòng đó là đáp án; nếu hết đỉnh để mở rộng mà không gặp ⇒ `-1`.

## 5. Cách 1 — Brute Force / baseline: vét cạn bằng quay lui (DFS)

**Ý tưởng.** Thử **mọi đường đi đơn** (không lặp lại gene) từ `startGene`: ở gene hiện tại, với mỗi gene `g` của `bank` chưa dùng mà `hamming == 1`, đi tiếp. Mỗi lần chạm `endGene`, cập nhật đáp án nhỏ nhất. Ta thêm **cắt nhánh**: nếu số bước hiện tại đã `>= best` thì dừng (không ảnh hưởng tính đúng).

Đây là cách "thử tất cả" dễ chứng minh đúng, nên dùng làm **oracle** cho các cách sau (với bank nhỏ).

In [8]:
def min_mutation_dfs(start, end, bank):
    if start == end:
        return 0
    if end not in bank:                      # gene cuối cùng bắt buộc phải hợp lệ
        return -1
    bank = list(dict.fromkeys(bank))         # bỏ trùng
    best = [float("inf")]

    def dfs(gene, steps, used):
        if steps >= best[0]:
            return
        if gene == end:
            best[0] = steps
            return
        for i, nxt in enumerate(bank):
            if i not in used and hamming(gene, nxt) == 1:
                used.add(i)
                dfs(nxt, steps + 1, used)
                used.remove(i)

    dfs(start, 0, set())
    return -1 if best[0] == float("inf") else best[0]


for s, e, bank, ans in examples:
    print(min_mutation_dfs(s, e, bank), "(kỳ vọng", ans, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Gọi `m = len(bank)`. Trường hợp xấu nhất thử mọi **hoán vị** các gene ⇒ Time **`O(m! · m · 8)`** (rất tệ nhưng `m ≤ 10`, và cắt nhánh giúp nhiều); Space `O(m)` ngăn xếp đệ quy.

**Nút thắt:** ta duyệt **đi sâu** và thử cả các đường dài vô ích. Vì mọi cạnh "nặng" như nhau (mỗi đột biến = 1), việc tìm đường **ít cạnh nhất** có thuật toán chuyên dụng: **BFS**, mỗi gene chỉ cần xử lý **một lần**.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** đáp án là **độ dài đường đi ngắn nhất** trên đồ thị không trọng số ⇒ **BFS từ `startGene`**, mở rộng theo từng vòng.

- Giữ `visited` để mỗi gene vào hàng đợi **tối đa một lần**.
- Khi lấy ra gene `== endGene`, số vòng hiện tại (độ sâu) chính là đáp án (vì BFS gặp mỗi đỉnh lần đầu bằng đường ngắn nhất).
- Hết hàng đợi mà không gặp `end` ⇒ `-1`.

Còn lại là chọn **cách liệt kê láng giềng**: quét cả `bank` (tối đa 10 phép so sánh, mỗi phép so 8 ký tự) hoặc **tự sinh 24 đột biến** rồi tra `set`. Và có thể tăng tốc nữa bằng cách chạy BFS từ **hai đầu**.

## 7. Các cách tiếp theo

### Cách 2 — BFS, quét ngân hàng để tìm láng giềng (lời giải chuẩn)

**Giải thích từng đoạn.**

1. `q = deque([(startGene, 0)])`: mỗi phần tử là `(gene, số đột biến đã dùng)`. `vis = {startGene}`.
2. Lấy `gene, depth = q.popleft()`. Nếu `gene == endGene` → trả `depth`.
3. Với mỗi `nxt` trong `bank`: nếu `hamming(gene, nxt) == 1` và `nxt not in vis` → `vis.add(nxt)`, `q.append((nxt, depth + 1))`.
4. Hết hàng đợi → `-1`.

(Nếu `start == end` thì lần lấy đầu tiên đã trả `0`, đúng như đề: điểm xuất phát tự hợp lệ.)

In [9]:
def min_mutation_bfs_scan(start, end, bank):
    q = deque([(start, 0)])
    vis = {start}
    while q:
        gene, depth = q.popleft()
        if gene == end:
            return depth
        for nxt in bank:
            if hamming(gene, nxt) == 1 and nxt not in vis:
                vis.add(nxt)
                q.append((nxt, depth + 1))
    return -1


for s, e, bank, ans in examples:
    print(min_mutation_bfs_scan(s, e, bank), "(kỳ vọng", ans, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


In [10]:
class Solution2:
    def minMutation(self, startGene: str, endGene: str, bank: list[str]) -> int:
        q = deque([(startGene, 0)])
        vis = {startGene}
        while q:
            gene, depth = q.popleft()
            if gene == endGene:
                return depth
            for nxt in bank:
                diff = sum(a != b for a, b in zip(gene, nxt))
                if diff == 1 and nxt not in vis:
                    vis.add(nxt)
                    q.append((nxt, depth + 1))
        return -1


print(Solution2().minMutation("AACCGGTT", "AAACGGTA", ["AACCGGTA", "AACCGCTA", "AAACGGTA"]))

2


**Độ phức tạp (Cách 2).** Mỗi gene vào hàng đợi tối đa một lần (`≤ m + 1` gene); mỗi lần ta quét `bank` (`m` phần tử) và so `L = 8` ký tự ⇒ Time **`O(m² · L)`** (`10² · 8 = 800` thao tác). Space **`O(m · L)`** cho hàng đợi và `vis`.

### Cách 3 — BFS, **sinh** láng giềng bằng cách thay từng ký tự

Thay vì quét `bank`, với gene hiện tại ta **tự tạo** `L × 3 = 24` đột biến (mục 3.5) và chỉ giữ cái **thuộc `bank_set`** và **chưa thăm**. Chi phí mỗi gene là `O(L · 3 · L)` (tạo chuỗi `L` ký tự) và **không phụ thuộc kích thước `bank`** — có lợi khi `bank` rất lớn (ở bài này `bank` chỉ có 10, nhưng đây là ý tưởng của Word Ladder).

**Giải thích từng đoạn.**

1. `start == end` → `0`. `bank_set = set(bank)`. Nếu `end not in bank_set` → `-1` (không có cách vào `end`).
2. BFS theo **vòng**: `for _ in range(len(q))` lấy trọn một vòng; sau mỗi vòng `steps += 1`.
3. Với mỗi gene, duyệt các đột biến; nếu `cand == end` → trả `steps`; nếu `cand in bank_set and cand not in vis` → thêm.

In [11]:
def min_mutation_bfs_gen(start, end, bank):
    if start == end:
        return 0
    bank_set = set(bank)
    if end not in bank_set:
        return -1
    q = deque([start])
    vis = {start}
    steps = 0
    while q:
        steps += 1
        for _ in range(len(q)):                      # trọn một vòng
            gene = q.popleft()
            for cand in neighbors(gene, bank_set):
                if cand == end:
                    return steps
                if cand not in vis:
                    vis.add(cand)
                    q.append(cand)
    return -1


for s, e, bank, ans in examples:
    print(min_mutation_bfs_gen(s, e, bank), "(kỳ vọng", ans, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


In [12]:
class Solution3:
    def minMutation(self, startGene: str, endGene: str, bank: list[str]) -> int:
        if startGene == endGene:
            return 0
        bank_set = set(bank)
        if endGene not in bank_set:
            return -1
        q = deque([startGene])
        vis = {startGene}
        steps = 0
        while q:
            steps += 1
            for _ in range(len(q)):
                gene = q.popleft()
                for i in range(len(gene)):
                    for c in "ACGT":
                        if c == gene[i]:
                            continue
                        cand = gene[:i] + c + gene[i + 1:]
                        if cand == endGene:
                            return steps
                        if cand in bank_set and cand not in vis:
                            vis.add(cand)
                            q.append(cand)
        return -1


print(Solution3().minMutation("AACCGGTT", "AACCGGTA", ["AACCGGTA"]))

1


**Độ phức tạp (Cách 3).** Có `≤ m + 1` gene; mỗi gene tạo `24` chuỗi, mỗi chuỗi tốn `O(L)` để tạo/băm ⇒ Time **`O(m · 24 · L) = O(m · L²)` hằng số nhỏ**, độc lập với việc quét `bank`. Space **`O(m · L)`**.

> Lưu ý: kiểm tra `cand == end` **trước** khi kiểm tra `cand in vis` là ổn vì `end` đã được xác nhận nằm trong `bank_set`, và `end != start`.

### Cách 4 — BFS **hai chiều**

Chạy đồng thời từ `start` và từ `end`. Giữ hai tập "mặt trận" `front` (từ `start`) và `back` (từ `end`); mỗi vòng **mở rộng tập nhỏ hơn** (để ít đỉnh nhất), nếu một láng giềng đã có trong tập bên kia ⇒ hai bên gặp nhau, số bước = số vòng đã mở rộng.

**Giải thích từng đoạn.**

1. Xử lý `start == end` → `0`; `end not in bank_set` → `-1`.
2. `front = {start}`, `back = {end}`, `visited = {start, end}`, `steps = 0`.
3. `while front and back`: `steps += 1`; nếu `len(front) > len(back)` thì **đổi chỗ** hai tập (ta luôn mở rộng tập nhỏ).
4. Với mỗi gene trong `front`, mỗi láng giềng `cand`: nếu `cand in back` → trả `steps`; nếu chưa thăm → thêm vào `nxt`.
5. `front = nxt`. Nếu `front` hoặc `back` rỗng → `-1`.

In [13]:
def min_mutation_bidirectional(start, end, bank):
    if start == end:
        return 0
    bank_set = set(bank)
    if end not in bank_set:
        return -1
    front, back = {start}, {end}
    visited = {start, end}
    steps = 0
    while front and back:
        steps += 1
        if len(front) > len(back):               # luôn mở rộng phía nhỏ hơn
            front, back = back, front
        nxt = set()
        for gene in front:
            for cand in neighbors(gene, bank_set):
                if cand in back:                 # hai mặt trận gặp nhau
                    return steps
                if cand not in visited:
                    visited.add(cand)
                    nxt.add(cand)
        front = nxt
    return -1


for s, e, bank, ans in examples:
    print(min_mutation_bidirectional(s, e, bank), "(kỳ vọng", ans, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


In [14]:
class Solution4:
    def minMutation(self, startGene: str, endGene: str, bank: list[str]) -> int:
        if startGene == endGene:
            return 0
        bank_set = set(bank)
        if endGene not in bank_set:
            return -1

        def neighbors(gene):
            for i in range(len(gene)):
                for c in "ACGT":
                    if c != gene[i]:
                        cand = gene[:i] + c + gene[i + 1:]
                        if cand in bank_set:
                            yield cand

        front, back = {startGene}, {endGene}
        visited = {startGene, endGene}
        steps = 0
        while front and back:
            steps += 1
            if len(front) > len(back):
                front, back = back, front
            nxt = set()
            for gene in front:
                for cand in neighbors(gene):
                    if cand in back:
                        return steps
                    if cand not in visited:
                        visited.add(cand)
                        nxt.add(cand)
            front = nxt
        return -1


print(Solution4().minMutation("AACCGGTT", "AAACGGTA", ["AACCGGTA", "AACCGCTA", "AAACGGTA"]))

2


**Độ phức tạp (Cách 4).** Trong trường hợp xấu nhất vẫn duyệt hết các đỉnh: Time **`O(m · L²)`**; nhưng với đồ thị "bung rộng" thì mỗi phía chỉ đi **một nửa** độ sâu nên số đỉnh duyệt giảm mạnh (từ `b^d` còn `2·b^(d/2)` với `b` là mức rẽ nhánh trung bình). Space **`O(m · L)`**. Với `m ≤ 10` lợi ích chưa thấy rõ, nhưng đây là kỹ thuật quan trọng cho Word Ladder (127).

### Cách 5 — Nới lỏng khoảng cách kiểu Bellman–Ford (quy hoạch động trên số bước)

Một góc nhìn **khác hẳn BFS**: giữ `dist[g]` = số đột biến ít nhất **đã biết** để tới `g` (ban đầu `dist[start] = 0`, còn lại `∞`). Lặp lại: với mọi cặp `(u, v)` kề nhau (`hamming == 1`), nếu `dist[u] + 1 < dist[v]` thì **cập nhật** `dist[v]`. Sau tối đa `|V|` vòng (đường đi ngắn nhất có tối đa `|V| − 1` cạnh), `dist` ổn định; trả `dist[end]`.

**Giải thích từng đoạn.**

1. `nodes = [start] + bank` (bỏ trùng, giữ thứ tự bằng `dict.fromkeys`).
2. `dist = {g: INF}`; `dist[start] = 0`.
3. Lặp tối đa `len(nodes)` lần: duyệt mọi `u` có `dist[u] < INF` và mọi `v`; nếu kề và `dist[u] + 1 < dist[v]` → cập nhật, đánh dấu `changed`.
4. Nếu một vòng không đổi gì → dừng sớm. Trả `dist[end]` (nếu `end` không thuộc `nodes` hoặc còn `INF` → `-1`).

Cách này **không** cần hàng đợi hay `visited`, chỉ dựa vào "cải thiện dần".

In [15]:
def min_mutation_relax(start, end, bank):
    nodes = list(dict.fromkeys([start] + list(bank)))
    INF = float("inf")
    dist = {g: INF for g in nodes}
    dist[start] = 0
    for _ in range(len(nodes)):
        changed = False
        for u in nodes:
            if dist[u] == INF:
                continue
            for v in nodes:
                if hamming(u, v) == 1 and dist[u] + 1 < dist[v]:
                    dist[v] = dist[u] + 1
                    changed = True
        if not changed:
            break
    if end not in dist or dist[end] == INF:
        return -1
    return dist[end]


for s, e, bank, ans in examples:
    print(min_mutation_relax(s, e, bank), "(kỳ vọng", ans, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


In [16]:
class Solution5:
    def minMutation(self, startGene: str, endGene: str, bank: list[str]) -> int:
        nodes = list(dict.fromkeys([startGene] + bank))
        INF = float("inf")
        dist = {g: INF for g in nodes}
        dist[startGene] = 0
        for _ in range(len(nodes)):
            changed = False
            for u in nodes:
                if dist[u] == INF:
                    continue
                for v in nodes:
                    diff = sum(a != b for a, b in zip(u, v))
                    if diff == 1 and dist[u] + 1 < dist[v]:
                        dist[v] = dist[u] + 1
                        changed = True
            if not changed:
                break
        return -1 if dist.get(endGene, INF) == INF else dist[endGene]


print(Solution5().minMutation("AAAAACCC", "AACCCCCC", ["AAAACCCC", "AAACCCCC", "AACCCCCC"]))

3


**Độ phức tạp (Cách 5).** `V = m + 1` đỉnh; mỗi vòng xét `V²` cặp, mỗi cặp so `L` ký tự; tối đa `V` vòng ⇒ Time **`O(V³ · L)`** (`11³ · 8 ≈ 10^4` — vẫn nhỏ). Space **`O(V)`**. Chậm hơn BFS nhưng minh họa ý tưởng "**nới lỏng**" (relaxation) dùng cho đồ thị có **trọng số** (Bellman–Ford) — nếu cạnh có trọng số khác nhau thì BFS thuần không còn đúng.

## 8. TRACE từng bước

Trace **Cách 3** (BFS sinh láng giềng) trên Ví dụ 2: in từng vòng, hàng đợi, gene lấy ra, các láng giềng tìm được và cái nào được thêm.

In [17]:
def trace_bfs(start, end, bank):
    if start == end:
        print("start == end -> 0")
        return 0
    bank_set = set(bank)
    if end not in bank_set:
        print("end không nằm trong bank -> -1")
        return -1
    q = deque([start])
    vis = {start}
    steps = 0
    while q:
        steps += 1
        print(f"=== vòng {steps}: hàng đợi = {list(q)}")
        for _ in range(len(q)):
            gene = q.popleft()
            nbs = neighbors(gene, bank_set)
            print(f"  lấy {gene}; láng giềng trong bank = {nbs}")
            for cand in nbs:
                if cand == end:
                    print(f"  -> gặp end {cand} sau {steps} đột biến")
                    return steps
                if cand not in vis:
                    vis.add(cand)
                    q.append(cand)
                    print(f"     thêm {cand} (chưa thăm)")
                else:
                    print(f"     bỏ  {cand} (đã thăm)")
    print("hết hàng đợi -> -1")
    return -1


s, e, bank, _ = examples[1]
assert trace_bfs(s, e, bank) == 2

=== vòng 1: hàng đợi = ['AACCGGTT']
  lấy AACCGGTT; láng giềng trong bank = ['AACCGGTA']
     thêm AACCGGTA (chưa thăm)
=== vòng 2: hàng đợi = ['AACCGGTA']
  lấy AACCGGTA; láng giềng trong bank = ['AAACGGTA', 'AACCGCTA']
  -> gặp end AAACGGTA sau 2 đột biến


Bảng trạng thái (Ví dụ 2):

| Vòng | Hàng đợi đầu vòng | Gene lấy ra | Láng giềng trong `bank` | Hành động |
|---|---|---|---|---|
| 1 | `[AACCGGTT]` | `AACCGGTT` | `AACCGGTA` | chưa thăm → thêm vào hàng đợi |
| 2 | `[AACCGGTA]` | `AACCGGTA` | `AACCGCTA`, `AAACGGTA` | `AACCGCTA` thêm; `AAACGGTA == end` → **trả `2`** |

Chú ý thứ tự duyệt láng giềng (theo vị trí `i` rồi theo `A, C, G, T`) có thể khiến ta gặp `end` trước hay sau `AACCGCTA`, nhưng **số vòng** thì không đổi.

Trace thêm một trường hợp **không có đường**: `bank = [AACCGGTA]`, `end = AAAAAAAA` (không thuộc bank) và một trường hợp bank có nhưng bị "đứt đoạn":

In [18]:
print("--- end không trong bank")
assert trace_bfs("AACCGGTT", "AAAAAAAA", ["AACCGGTA"]) == -1
print("--- end trong bank nhưng không có đường nối")
assert trace_bfs("AACCGGTT", "AAAAAAAA", ["AACCGGTA", "AAAAAAAA"]) == -1

--- end không trong bank
end không nằm trong bank -> -1
--- end trong bank nhưng không có đường nối
=== vòng 1: hàng đợi = ['AACCGGTT']
  lấy AACCGGTT; láng giềng trong bank = ['AACCGGTA']
     thêm AACCGGTA (chưa thăm)
=== vòng 2: hàng đợi = ['AACCGGTA']
  lấy AACCGGTA; láng giềng trong bank = []
hết hàng đợi -> -1


## 9. Code nộp LeetCode

Bản nộp (Cách 2: BFS quét ngân hàng — ngắn và đủ nhanh vì `bank ≤ 10`). Trên LeetCode cần `from collections import deque` (thường đã có sẵn).

In [19]:
class Solution:
    def minMutation(self, startGene: str, endGene: str, bank: list[str]) -> int:
        q = deque([(startGene, 0)])
        vis = {startGene}
        while q:
            gene, depth = q.popleft()
            if gene == endGene:
                return depth
            for nxt in bank:
                diff = sum(a != b for a, b in zip(gene, nxt))
                if diff == 1 and nxt not in vis:
                    vis.add(nxt)
                    q.append((nxt, depth + 1))
        return -1


for s, e, bank, ans in examples:
    out = Solution().minMutation(s, e, bank)
    print(out)
    assert out == ans

1
2
3


## 10. Vì sao thuật toán đúng

**BFS (Cách 2, 3).** Bất biến: *các gene được lấy ra khỏi hàng đợi theo thứ tự **độ sâu không giảm**, và gene lấy ra với độ sâu `d` có khoảng cách ngắn nhất từ `start` đúng bằng `d`.* Chứng minh quy nạp theo `d`: ban đầu `start` có `d = 0`. Mọi gene cách `start` đúng `d + 1` đột biến có một đường ngắn nhất đi qua một gene cách `d`; gene đó được lấy ra (do quy nạp) và khi quét/ sinh láng giềng sẽ thêm gene này vào hàng đợi với độ sâu `d + 1` nếu nó chưa được thêm — mà nếu đã được thêm thì cũng với độ sâu **không lớn hơn** `d + 1` (vì BFS xử lý theo thứ tự độ sâu), và bằng khoảng cách thật (không nhỏ hơn). `visited` chỉ loại bỏ các lần thêm lặp, không làm mất đường ngắn nhất. Khi lấy ra `end` lần đầu, độ sâu của nó là khoảng cách ngắn nhất. Hết hàng đợi ⇒ `end` không với tới được ⇒ `-1`.

**Điều kiện hợp lệ.** Chỉ các gene trong `bank` được đưa vào hàng đợi (ngoài `start`), nên đường đi tìm được chỉ qua gene hợp lệ, và `end` chỉ tới được nếu thuộc `bank` (trừ `start == end` ⇒ `0`).

**Hai chiều (Cách 4).** Mỗi vòng mở rộng một phía thêm đúng một cạnh; khi một láng giềng của phía này nằm trong "mặt trận" phía kia, tổng số cạnh là `(số vòng)`. Vì đồ thị **vô hướng** (đột biến thuận nghịch: nếu `u→v` hợp lệ do khác một ký tự thì `v→u` cũng thế, mọi gene trong bank hợp lệ) nên đi từ `end` ngược lại là hợp lệ.

**Cách 5** là Bellman–Ford: sau `k` vòng, `dist[v]` ≤ độ dài đường ngắn nhất dùng tối đa `k` cạnh; do đường ngắn nhất không lặp đỉnh, `|V|` vòng là đủ. **Cách 1** thử mọi đường đơn nên hiển nhiên tìm được đường ngắn nhất.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Quay lui DFS (có cắt nhánh) | `O(m! · m · L)` tệ nhất | `O(m)` |
| 2. BFS quét bank | `O(m² · L)` | `O(m · L)` |
| 3. BFS sinh láng giềng | `O(m · L²)` (hằng số 24) | `O(m · L)` |
| 4. BFS hai chiều | `O(m · L²)` (thường ít đỉnh hơn) | `O(m · L)` |
| 5. Nới lỏng kiểu Bellman–Ford | `O(m³ · L)` | `O(m)` |

`m = len(bank) ≤ 10`, `L = 8`. Với giới hạn đề mọi cách (trừ vét cạn trong trường hợp xấu nhất) đều chạy tức thì; BFS là lựa chọn đúng bản chất (đường ngắn nhất, đồ thị không trọng số).

## 12. Bộ test

Ta dùng một **oracle độc lập** `oracle_floyd` (Floyd–Warshall trên ma trận kề) để đối chiếu, vì nó không dùng hàng đợi/BFS/DFS: tính khoảng cách mọi cặp đỉnh rồi lấy `dist[start][end]`.

In [20]:
import random, itertools

def oracle_floyd(start, end, bank):
    if start == end:
        return 0
    nodes = list(dict.fromkeys([start] + list(bank)))
    if end not in nodes:
        return -1
    n = len(nodes)
    INF = 10 ** 9
    d = [[0 if i == j else (1 if hamming(nodes[i], nodes[j]) == 1 else INF) for j in range(n)] for i in range(n)]
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if d[i][k] + d[k][j] < d[i][j]:
                    d[i][j] = d[i][k] + d[k][j]
    r = d[0][nodes.index(end)]
    return -1 if r >= INF else r


ALL = {
    "cach1": min_mutation_dfs,
    "cach2": min_mutation_bfs_scan,
    "cach3": min_mutation_bfs_gen,
    "cach4": min_mutation_bidirectional,
    "cach5": min_mutation_relax,
    "nop":   lambda s, e, b: Solution().minMutation(s, e, b),
}

def run_all(start, end, bank, expected=None, skip_dfs=False):
    want = oracle_floyd(start, end, bank) if expected is None else expected
    for name, fn in ALL.items():
        if skip_dfs and name == "cach1":
            continue
        got = fn(start, end, list(bank))
        assert got == want, (name, start, end, bank, got, want)
    return want

for s, e, bank, ans in examples:
    assert run_all(s, e, bank, ans) == ans
print("ví dụ của đề: OK")

ví dụ của đề: OK


### 12.1. Các trường hợp biên

In [21]:
# start == end: 0 (dù bank rỗng hoặc end không trong bank)
print(run_all("AACCGGTT", "AACCGGTT", [], 0), run_all("AACCGGTT", "AACCGGTT", ["AACCGGTA"], 0))
# bank rỗng, start != end -> -1
print(run_all("AACCGGTT", "AACCGGTA", [], -1))
# end không nằm trong bank (dù cách start đúng 1 ký tự) -> -1
print(run_all("AACCGGTT", "AACCGGTA", ["AACCGGTC"], -1))
# end trong bank nhưng không có đường nối (đứt đoạn)
print(run_all("AAAAAAAA", "CCCCCCCC", ["AAAAAAAC", "CCCCCCCC"], -1))
# start nằm trong bank (cho phép) và bank có trùng lặp
print(run_all("AAAAAAAA", "AAAAAACC", ["AAAAAAAA", "AAAAAAAC", "AAAAAAAC", "AAAAAACC"], 2))
# chỉ qua đúng một đường dài tối đa: bank toàn 10 gene tạo chuỗi dài
chain = []
g = list("AAAAAAAA")
for i in range(8):
    g[i] = "C"
    chain.append("".join(g))
chain.append("".join(["G"] + ["C"] * 7))
chain.append("".join(["G", "G"] + ["C"] * 6))
print(len(chain), run_all("AAAAAAAA", chain[-1], chain, 10))
# một đường tắt: hai đường song song, BFS phải chọn ngắn hơn
print(run_all("AAAAAAAA", "AAAAAACC", ["AAAAAAAC", "AAAAAACC", "CAAAAAAA", "CAAAAAAC", "CAAAAACC"], 2))
print("OK")

0 0
-1
-1
-1
2
10 10
2
OK


### 12.2. Phản ví dụ cho các cách làm sai

Ba hàm sai điển hình (giải thích ở mục 13); test cho thấy chúng cho đáp án sai:

In [22]:
def wrong_first_path_dfs(start, end, bank):      # DFS, trả về đường ĐẦU TIÊN tìm được
    if start == end:
        return 0
    def dfs(gene, steps, used):
        if gene == end:
            return steps
        for i, nxt in enumerate(bank):
            if i not in used and hamming(gene, nxt) == 1:
                r = dfs(nxt, steps + 1, used | {i})
                if r != -1:
                    return r
        return -1
    return dfs(start, 0, frozenset())

def wrong_ignore_bank_end(start, end, bank):     # chỉ nhìn hamming(start, end), bỏ qua bank
    if start == end:
        return 0
    if hamming(start, end) == 1:
        return 1
    return min_mutation_bfs_scan(start, end, bank)

def wrong_off_by_one(start, end, bank):          # đếm số gene trên đường, không phải số đột biến
    r = min_mutation_bfs_scan(start, end, bank)
    return r if r == -1 else r + 1

long_first = ("AAAAAAAA", "AAAAAACC", ["CAAAAAAA", "CAAAAAAC", "CAAAAACC", "AAAAAAAC", "AAAAAACC"])
print("DFS đầu tiên:", wrong_first_path_dfs(*long_first), "| đúng:", oracle_floyd(*long_first))
assert wrong_first_path_dfs(*long_first) != oracle_floyd(*long_first)

print("bỏ qua bank:", wrong_ignore_bank_end("AACCGGTT", "AACCGGTA", []), "| đúng:", oracle_floyd("AACCGGTT", "AACCGGTA", []))
assert wrong_ignore_bank_end("AACCGGTT", "AACCGGTA", []) != -1

print("lệch 1:", wrong_off_by_one("AACCGGTT", "AACCGGTA", ["AACCGGTA"]), "| đúng: 1")
assert wrong_off_by_one("AACCGGTT", "AACCGGTA", ["AACCGGTA"]) == 2

DFS đầu tiên: 4 | đúng: 2
bỏ qua bank: 1 | đúng: -1
lệch 1: 2 | đúng: 1


### 12.3. Đối chiếu năm cách với oracle Floyd–Warshall trên dữ liệu ngẫu nhiên (seed cố định)

Sinh dữ liệu "có cấu trúc" (để có đường đi dài chứ không toàn `-1`): chọn `start` ngẫu nhiên; mỗi gene mới của `bank` là **đột biến 1–2 ký tự của một gene đã có** (start hoặc bank trước đó) — nên đồ thị thường liên thông; thêm vài gene hoàn toàn ngẫu nhiên. `end` lấy ngẫu nhiên từ bank, hoặc ngẫu nhiên hẳn. Với `bank` nhỏ (≤ 7) ta cũng chạy Cách 1 (vét cạn) để chắc chắn.

In [23]:
def mutate(g, k, rng):
    g = list(g)
    for i in rng.sample(range(8), k):
        g[i] = rng.choice([c for c in "ACGT" if c != g[i]])
    return "".join(g)

def rand_gene(rng):
    return "".join(rng.choice("ACGT") for _ in range(8))

rng = random.Random(433)
stats = {"-1": 0, "0": 0, "pos": 0}
for it in range(500):
    start = rand_gene(rng)
    m = rng.randint(0, 10)
    pool = [start]
    bank = []
    for _ in range(m):
        if rng.random() < 0.15:
            g = rand_gene(rng)
        else:
            g = mutate(rng.choice(pool), rng.choice([1, 1, 1, 2]), rng)
        bank.append(g)
        pool.append(g)
    if bank and rng.random() < 0.8:
        end = rng.choice(bank)
    else:
        end = rng.choice([start, rand_gene(rng)])
    ans = run_all(start, end, bank, skip_dfs=len(bank) > 7)
    stats["-1" if ans == -1 else "0" if ans == 0 else "pos"] += 1
print("500 bộ dữ liệu ngẫu nhiên: OK;", stats)
assert stats["pos"] > 100 and stats["-1"] > 20

500 bộ dữ liệu ngẫu nhiên: OK; {'-1': 238, '0': 83, 'pos': 179}


### 12.4. Kiểm tra tính chất: cạnh đối xứng, đáp án không đổi khi đổi thứ tự/ trùng lặp trong bank

Hai tính chất cần có: (1) đáp án **không phụ thuộc thứ tự** các gene trong `bank` và việc có gene **trùng** hay không; (2) nếu `end` và `start` **đổi chỗ** (và `start` được đưa vào `bank` để hợp lệ), đáp án vẫn như cũ — vì đột biến là **đối xứng**.

In [24]:
rng = random.Random(7)
checked = 0
for _ in range(300):
    start = rand_gene(rng)
    pool, bank = [start], []
    for _ in range(rng.randint(1, 10)):
        g = mutate(rng.choice(pool), rng.choice([1, 1, 2]), rng)
        bank.append(g); pool.append(g)
    end = rng.choice(bank)
    base = Solution().minMutation(start, end, bank)

    shuffled = bank[:]
    rng.shuffle(shuffled)
    assert Solution().minMutation(start, end, shuffled) == base                    # (1) thứ tự
    assert Solution().minMutation(start, end, bank + bank[:3]) == base             # (1) trùng lặp

    if len(bank) < 10 and start not in bank:                                       # (2) đối xứng
        rev = Solution().minMutation(end, start, bank + [start])
        assert rev == base, (start, end, bank, rev, base)
    checked += 1
print("tính chất bất biến trên", checked, "bộ dữ liệu: OK")

tính chất bất biến trên 300 bộ dữ liệu: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `visited` — BFS lặp vô hạn (hoặc lặp thừa)

In [25]:
def wrong_no_visited(start, end, bank, limit=200):
    q = deque([(start, 0)])
    pops = 0
    while q and pops < limit:                 # limit chỉ để demo không treo máy
        gene, depth = q.popleft()
        pops += 1
        if gene == end:
            return depth, pops
        for nxt in bank:
            if hamming(gene, nxt) == 1:       # THIẾU: and nxt not in vis
                q.append((nxt, depth + 1))
    return -1, pops

print(wrong_no_visited("AAAAAAAA", "GGGGGGGG", ["AAAAAAAC", "AAAAAACC", "AAAAAAAA"]))
print("đúng:", Solution().minMutation("AAAAAAAA", "GGGGGGGG", ["AAAAAAAC", "AAAAAACC", "AAAAAAAA"]))

(-1, 200)
đúng: -1


**Vì sao sai:** đồ thị đột biến có **chu trình** (`A ⇄ B ⇄ A`; đột biến đi được cả hai chiều). Không đánh dấu "đã thăm" thì gene bị đưa lại vào hàng đợi mãi: chương trình chạy đến hết giới hạn (ở đây ta chặn bằng `limit`) khi `end` không với tới được. **Đúng:** đánh dấu ngay lúc **đưa vào** hàng đợi (`vis.add(nxt)`), như Cách 2.

### Lỗi 2: Nhầm điều kiện "khác đúng một ký tự" (`<= 1`, `>= 1`)

In [26]:
def wrong_diff_le1(start, end, bank):
    q = deque([(start, 0)]); vis = {start}
    while q:
        gene, depth = q.popleft()
        if gene == end:
            return depth
        for nxt in bank:
            diff = sum(a != b for a, b in zip(gene, nxt))
            if diff <= 2 and nxt not in vis:          # SAI: cho phép đổi tới 2 ký tự một lần
                vis.add(nxt); q.append((nxt, depth + 1))
    return -1

case = ("AAAAACCC", "AACCCCCC", ["AAAACCCC", "AAACCCCC", "AACCCCCC"])
print("sai (cho phép đổi 2 ký tự):", wrong_diff_le1(*case), "| đúng:", Solution().minMutation(*case))
assert wrong_diff_le1(*case) != Solution().minMutation(*case)

sai (cho phép đổi 2 ký tự): 2 | đúng: 3


**Vì sao sai:** một lần đột biến là đổi **đúng một** ký tự, nên điều kiện là `diff == 1`. Cho phép `diff == 2` làm ta "nhảy cóc" hai ký tự một bước và đếm **ít** đột biến hơn thực tế. (Còn `diff == 0` là chính gene đó — vô nghĩa, và bị `visited` chặn.)

### Lỗi 3: Quên kiểm tra `endGene` có nằm trong `bank`

In [27]:
print("sai (chỉ nhìn start/end):", wrong_ignore_bank_end("AACCGGTT", "AACCGGTA", []))
print("đúng                  :", Solution().minMutation("AACCGGTT", "AACCGGTA", []))

sai (chỉ nhìn start/end): 1
đúng                  : -1


**Vì sao sai:** đề nói *mọi* gene mới — kể cả `endGene` — phải nằm trong `bank`. Cách sinh láng giềng nếu không lọc bằng `bank_set` (hoặc so sánh trực tiếp `start/end`) sẽ cho đáp án `1` dù `bank` rỗng. **Đúng:** chỉ chấp nhận gene thuộc `bank` (Cách 2 tự đúng vì chỉ đưa phần tử của `bank` vào hàng đợi; Cách 3, 4 kiểm tra `end in bank_set` ngay đầu).

### Lỗi 4: Đếm lệch 1 hoặc quên trường hợp `start == end`

In [28]:
print("đếm lệch:", wrong_off_by_one("AACCGGTT", "AACCGGTA", ["AACCGGTA"]), "(đúng 1)")

def wrong_no_equal_case(start, end, bank):
    bank_set = set(bank)
    if end not in bank_set:
        return -1                               # start == end nhưng end không trong bank -> trả sai -1
    return min_mutation_bfs_gen(start, end, bank)

print("start == end, bank rỗng:", wrong_no_equal_case("AACCGGTT", "AACCGGTT", []), "| đúng: 0")

đếm lệch: 2 (đúng 1)
start == end, bank rỗng: -1 | đúng: 0


**Vì sao sai:** số đột biến = **số cạnh**, không phải số đỉnh trên đường (đỉnh nhiều hơn cạnh `1`). Và điểm xuất phát tự hợp lệ nên `start == end` ⇒ `0` mà **không cần** `end` thuộc `bank`. **Đúng:** trả `0` ngay (hoặc để BFS lấy ra `start` với `depth = 0`).

### Lỗi 5: Dùng DFS và trả về đường đầu tiên tìm thấy

In [29]:
print("DFS đầu tiên:", wrong_first_path_dfs(*long_first), "(đường dài)")
print("BFS         :", Solution().minMutation(*long_first), "(ngắn nhất)")

DFS đầu tiên: 4 (đường dài)
BFS         : 2 (ngắn nhất)


**Vì sao sai:** DFS đi sâu theo thứ tự `bank`, gặp `end` bằng một đường **dài** trước khi thử đường tắt. "Ít nhất" trên đồ thị không trọng số ⇒ **BFS** (hoặc DFS vét cạn rồi lấy `min`, như Cách 1).

## 14. Mẹo & nhận dạng pattern

- **"Số bước/biến đổi ít nhất", mỗi bước có chi phí như nhau ⇒ BFS** trên **đồ thị ẩn** (trạng thái = đỉnh, bước đi = cạnh).
- Khi bài cho "chuỗi biến đổi từng ký tự" (gene, word ladder, ổ khóa số, trạng thái bảng) ⇒ nghĩ ngay *đồ thị ẩn + BFS*. Cùng họ: *Word Ladder (127)*, *Open the Lock (752)*, *Sliding Puzzle (773)*.
- **Tra cứu/đánh dấu** bằng `set`, không `list`.
- **Hai cách liệt kê láng giềng:** (a) quét hết kho (tốt khi kho nhỏ), (b) sinh biến thể rồi tra `set` (tốt khi kho lớn, không gian biến thể nhỏ).
- **Đánh dấu `visited` lúc đưa vào hàng đợi**, không phải lúc lấy ra — tránh đưa trùng.
- **BFS hai chiều** khi biết cả hai đầu và độ rẽ nhánh lớn.
- Kiểm tra các **trường hợp đặc biệt đề nêu ngầm**: `start == end`, `end` không trong bank, bank rỗng, `start` có thể không thuộc bank.

## 15. Biến thể

1. **Word Ladder (127):** y hệt nhưng ký tự là chữ cái thường (`26` thay vì `4`) và `bank` lớn; trả **độ dài** dãy (số đỉnh, = số cạnh + 1).
2. **Word Ladder II (126):** liệt kê **mọi** đường ngắn nhất ⇒ BFS lưu cha + DFS dựng lại.
3. **Trả về chính đường đi** (danh sách gene) thay vì chỉ số bước — lưu `parent` (bài tập 4).
4. **Đếm số đường đột biến ngắn nhất** (bài tập 5a).
5. **Open the Lock (752):** trạng thái là 4 chữ số, mỗi bước quay một bánh ±1, có "ô cấm".
6. **Chi phí mỗi đột biến khác nhau:** BFS không còn đúng ⇒ **Dijkstra** (hoặc Bellman–Ford như Cách 5).
7. **Gene dài hơn / bảng chữ cái khác:** vẫn BFS; chọn sinh láng giềng hay quét kho tùy kích thước `bank`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui DFS | thử mọi đường đơn, lấy min | `O(m!·m·L)` tệ nhất | `O(m)` | oracle, bank rất nhỏ |
| 2. BFS quét bank | láng giềng = gene trong bank khác 1 ký tự | `O(m²·L)` | `O(m·L)` | **nên nộp** (ngắn gọn) |
| 3. BFS sinh láng giềng | thay từng ký tự (24 ứng viên), tra `set` | `O(m·L²)` | `O(m·L)` | bank lớn, không gian biến thể nhỏ |
| 4. BFS hai chiều | mở rộng từ hai đầu, gặp nhau ở giữa | `O(m·L²)` (ít đỉnh hơn) | `O(m·L)` | đồ thị rộng, biết cả hai đầu |
| 5. Nới lỏng kiểu Bellman–Ford | cập nhật `dist` đến khi ổn định | `O(m³·L)` | `O(m)` | trọng số cạnh khác nhau |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `hamming("AAAAAAAA", "AAAACCCC")`; (b) `len(neighbors("AAAAAAAA", {"AAAAAAAC", "AAAAAACC", "CAAAAAAA"}))`; (c) `Solution().minMutation("AACCGGTT", "AACCGGTT", [])`; (d) `Solution().minMutation("AAAAAAAA", "CCCCCCCC", ["CCCCCCCC"])`.

**Bài 2 (tính tay).** Với `start = "AAAAACCC"`, `end = "AACCCCCC"`, `bank = ["AAAACCCC", "AAACCCCC", "AACCCCCC"]`: vẽ đồ thị (đỉnh, cạnh), viết hàng đợi ở từng vòng BFS và đáp án.

**Bài 3 (điền TODO).** Hoàn thành `min_mutation_todo(start, end, bank)` theo Cách 2.

In [30]:
def min_mutation_todo(start, end, bank):
    # TODO 1: q = deque([(start, 0)]); vis = {start}
    # TODO 2: while q: gene, depth = q.popleft()
    # TODO 3: nếu gene == end thì return depth
    # TODO 4: với mỗi nxt trong bank: nếu hamming(gene, nxt) == 1 và nxt chưa thăm
    #         thì vis.add(nxt) và q.append((nxt, depth + 1))
    # TODO 5: hết hàng đợi -> return -1
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `min_mutation_path(start, end, bank)` trả **danh sách các gene** trên một đường đột biến ngắn nhất (gồm cả `start` và `end`), hoặc `[]` nếu không có. Gợi ý: BFS và lưu `parent[gene]`; dựng lại đường bằng cách đi ngược từ `end`. Trường hợp `start == end` trả `[start]`. Kiểm tra: độ dài đường `- 1 ==` kết quả của `Solution().minMutation`, và mỗi cặp liền kề khác đúng 1 ký tự.

**Bài 5 (biến thể / mini challenge).** (a) `count_shortest(start, end, bank)`: số **đường đột biến ngắn nhất** khác nhau (tính theo dãy gene); trả `0` nếu không có. Ví dụ: `start=AAAAAAAA`, `end=CCAAAAAA`, `bank=[CAAAAAAA, ACAAAAAA, CCAAAAAA]` có đúng `2` đường. (b) `reachable_count(start, bank)`: số gene trong `bank` (khác nhau, không tính `start`) **với tới được** từ `start`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `4`; (b) `AAAAAAAC` (1 khác) và `CAAAAAAA` (1 khác) là láng giềng, `AAAAAACC` (2 khác) thì không ⇒ `2`; (c) `0`; (d) `-1` (khác 8 ký tự, cần 8 bước nhưng không có gene trung gian). Bài 3: nhớ `vis.add(nxt)` ngay lúc đưa vào hàng đợi.
- **Mức 2 (vừa):** Bài 2: đồ thị là một đường thẳng `start – AAAACCCC – AAACCCCC – end`; hàng đợi: `[start]` → `[AAAACCCC]` → `[AAACCCCC]` → `[end]` ⇒ `3`. Bài 4: `parent = {start: None}`; khi thêm `nxt` thì `parent[nxt] = gene`.
- **Mức 3 (sâu):** Bài 5(a): BFS tính `ways[g]` = số đường ngắn nhất tới `g` và `dist[g]`; khi gặp láng giềng `v` với `dist[v] == dist[u] + 1` thì `ways[v] += ways[u]`. 5(b): BFS/DFS đơn giản, đếm số đỉnh thăm được trừ `start` (nếu `start` có trong bank thì không đếm nó).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert hamming("AAAAAAAA", "AAAACCCC") == 4
assert len(neighbors("AAAAAAAA", {"AAAAAAAC", "AAAAAACC", "CAAAAAAA"})) == 2
assert Solution().minMutation("AACCGGTT", "AACCGGTT", []) == 0
assert Solution().minMutation("AAAAAAAA", "CCCCCCCC", ["CCCCCCCC"]) == -1

# Bài 2
assert Solution().minMutation("AAAAACCC", "AACCCCCC", ["AAAACCCC", "AAACCCCC", "AACCCCCC"]) == 3


# Bài 3
def min_mutation_todo_answer(start, end, bank):
    q = deque([(start, 0)])
    vis = {start}
    while q:
        gene, depth = q.popleft()
        if gene == end:
            return depth
        for nxt in bank:
            if hamming(gene, nxt) == 1 and nxt not in vis:
                vis.add(nxt)
                q.append((nxt, depth + 1))
    return -1

for s, e, bank, ans in examples:
    assert min_mutation_todo_answer(s, e, bank) == ans


# Bài 4
def min_mutation_path(start, end, bank):
    if start == end:
        return [start]
    parent = {start: None}
    q = deque([start])
    while q:
        gene = q.popleft()
        for nxt in bank:
            if hamming(gene, nxt) == 1 and nxt not in parent:
                parent[nxt] = gene
                if nxt == end:
                    path = [end]
                    while parent[path[-1]] is not None:
                        path.append(parent[path[-1]])
                    return path[::-1]
                q.append(nxt)
    return []

for s, e, bank, ans in examples:
    path = min_mutation_path(s, e, bank)
    assert len(path) - 1 == ans and path[0] == s and path[-1] == e
    assert all(hamming(a, b) == 1 for a, b in zip(path, path[1:]))
    assert all(g in bank for g in path[1:])
assert min_mutation_path("AAAAAAAA", "CCCCCCCC", []) == []
assert min_mutation_path("AAAAAAAA", "AAAAAAAA", []) == ["AAAAAAAA"]


# Bài 5
def count_shortest(start, end, bank):
    nodes = list(dict.fromkeys([start] + list(bank)))
    if end not in nodes:
        return 0
    dist = {start: 0}
    ways = {start: 1}
    q = deque([start])
    while q:
        u = q.popleft()
        for v in nodes:
            if hamming(u, v) != 1:
                continue
            if v not in dist:
                dist[v] = dist[u] + 1
                ways[v] = ways[u]
                q.append(v)
            elif dist[v] == dist[u] + 1:
                ways[v] += ways[u]
    return ways.get(end, 0)

assert count_shortest("AAAAAAAA", "CCAAAAAA", ["CAAAAAAA", "ACAAAAAA", "CCAAAAAA"]) == 2
assert count_shortest("AACCGGTT", "AACCGGTA", ["AACCGGTA"]) == 1
assert count_shortest("AACCGGTT", "AACCGGTA", []) == 0


def reachable_count(start, bank):
    seen = {start}
    q = deque([start])
    while q:
        u = q.popleft()
        for v in bank:
            if hamming(u, v) == 1 and v not in seen:
                seen.add(v)
                q.append(v)
    return len(seen - {start})

assert reachable_count("AAAAAAAA", ["AAAAAAAC", "AAAAAACC", "GGGGGGGG"]) == 2
assert reachable_count("AAAAAAAA", []) == 0
print("Đáp án bài tập: OK")

Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (BFS đồ thị ẩn, đếm số cạnh)

```python
class Solution:
    def minMutation(self, startGene: str, endGene: str, bank: List[str]) -> int:
        q = deque([(startGene, 0)])
        vis = {startGene}
        while q:
            gene, depth = q.popleft()
            if gene == endGene:
                return depth
            for nxt in bank:
                diff = sum(a != b for a, b in zip(gene, nxt))
                if diff == 1 and nxt not in vis:
                    vis.add(nxt)
                    q.append((nxt, depth + 1))
        return -1
```

**Checklist**

- [ ] Nhận ra **đồ thị ẩn** (đỉnh = gene, cạnh = khác đúng 1 ký tự) và bài toán là **đường ngắn nhất không trọng số ⇒ BFS**?
- [ ] Điều kiện cạnh là `diff == 1` (không `<= 1`, không `<= 2`)?
- [ ] Có `visited` (đánh dấu lúc đưa vào hàng đợi) để không lặp vô hạn?
- [ ] Chỉ đi qua gene **thuộc `bank`**; `endGene` cũng phải thuộc `bank` (trừ `start == end ⇒ 0`)?
- [ ] Số đột biến = độ sâu BFS (số cạnh), không phải số đỉnh?
- [ ] Hết hàng đợi ⇒ trả `-1`?
- [ ] Đã thử: `start == end`, bank rỗng, `end` không trong bank, `start` trong bank, bank có phần tử trùng, đường dài `10` bước?
- [ ] Dùng `deque.popleft()` (không `list.pop(0)`), và `set` cho tra cứu?